# URA Hackathon 2026 — End-to-End Reproduction of `v10_calibrated` (Public LB 0.6685)

**Pipeline:** Fine-tuned VietOCR (`vgg_transformer` + CRAFT detector) for `ocr_text`  →  `CalibratedRuleHead` for `product_name`.

This single notebook reproduces our best **single-model** submission, exactly as scored on the Kaggle public leaderboard (**0.6685**). It is self-contained: every component (scoring, OCR, product head, empty-gate, cross-validation, submission builder) is embedded inline so it runs on Kaggle with no external `src/` package.

**Score metric (fixed by competition):**

$$\text{Score} = 0.6 \cdot F1_{\text{product}} + 0.4 \cdot (1 - \text{CER})$$

| Stage | Section | Output |
|:--|:--|:--|
| 0. Setup & config | §0 | paths, toggles |
| 1. Data understanding | §1 | labels, test ids |
| 2. OCR engine (VietOCR-FT) | §2 | `ocr_text` per image |
| 3. Product head (CalibratedRuleHead) | §3 | embedded source |
| 4. Cross-validation (5-fold GroupKFold) | §4 | **CV composite 0.6142** |
| 5. Train final head | §5 | head fit on all labels |
| 6. Inference + empty-gate | §6 | test predictions |
| 7. Build & validate `submission.csv` | §7 | competition file |

> **Two ways to run the OCR stage (§2):**
> - **Fast (recommended for review):** attach our precomputed OCR parquet dataset (`ocr_vietocr_ft_all.parquet`, `ocr_vietocr_ft_test.parquet`) and keep `RUN_OCR_FROM_SCRATCH = False`. The notebook loads them and runs the rest in ~2 minutes.
> - **Full (from raw images):** set `RUN_OCR_FROM_SCRATCH = True` with **GPU T4 + Internet ON**. This fine-tunes VietOCR and runs inference (~2–4 h), then continues automatically.


## §0  Setup, config, and toggles

Auto-detects the competition data mount (`/kaggle/input/.../test.csv`) and any attached OCR-cache dataset. The two metric weights (0.6 / 0.4) and all head hyper-parameters are fixed here to the exact values that produced `v10_calibrated`.

In [ ]:
import os, re, csv, hashlib, unicodedata, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ── Metric weights (FIXED by competition) ───────────────────────────────────────
W_PRODUCT_F1 = 0.6
W_OCR        = 0.4
MAX_OCR_LEN  = 500

# ── Product-head hyper-parameters (the v10_calibrated config) ───────────────────
MIN_PPROD       = 0.55   # only deploy a family rule if its train precision >= 0.55
GATE_THRESHOLD  = 0.75   # long-tail classifier "has-product" gate
MIN_CLASS_COUNT = 12     # min support for a classifier class
EMPTY_GATE_THR  = 0.60   # empty-gate: zero OCR text when P(textless) >= this
N_FOLDS         = 5      # GroupKFold splits for CV

# ── Run mode ────────────────────────────────────────────────────────────────────
# False = load precomputed OCR parquet (fast, ~2 min). True = fine-tune VietOCR from
# raw images (needs GPU + Internet, ~2-4 h), then continue automatically.
RUN_OCR_FROM_SCRATCH = False

OCR_ENGINE_TAG = "vietocr_ft"   # naming for cache files

# ── Locate competition data mount ───────────────────────────────────────────────
def _find_competition_root():
    for base in ("/kaggle/input", "."):
        b = Path(base)
        if not b.exists():
            continue
        for p in b.rglob("test.csv"):
            if (p.parent / "train_labels.csv").exists():
                return p.parent
    raise FileNotFoundError("Could not find competition data (test.csv + train_labels.csv).")

ROOT = _find_competition_root()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
print("Competition ROOT :", ROOT)
print("Working  dir     :", WORK)

# ── Locate attached OCR-cache parquets (if any) ─────────────────────────────────
def _find_parquet(stem):
    """Search /kaggle/input and working dir for ocr_vietocr_ft_<split>.parquet."""
    names = [f"ocr_{OCR_ENGINE_TAG}_{stem}.parquet", f"{stem}.parquet"]
    for base in ("/kaggle/input", "/kaggle/working", "."):
        b = Path(base)
        if not b.exists():
            continue
        for nm in names:
            hits = list(b.rglob(nm))
            if hits:
                return hits[0]
    return None

OCR_ALL_PARQUET  = _find_parquet("all")
OCR_TEST_PARQUET = _find_parquet("test")
print("OCR cache (all)  :", OCR_ALL_PARQUET)
print("OCR cache (test) :", OCR_TEST_PARQUET)

if not RUN_OCR_FROM_SCRATCH and (OCR_ALL_PARQUET is None or OCR_TEST_PARQUET is None):
    print("\n[!] No precomputed OCR parquet found. Either attach the cache dataset")
    print("    or set RUN_OCR_FROM_SCRATCH = True to fine-tune VietOCR from images.")

## §1  Data understanding & the exact metric

We load `train_labels.csv` (image_id, ocr_text, product_name) and `test.csv` (image_id). Empty strings are preserved (not NaN) — an empty `product_name` is a valid label (~41% of train rows have no product). The competition metric is re-implemented exactly so our CV measures the same thing Kaggle scores.

In [ ]:
# ── Load labels & test ids (empty strings preserved, not NaN) ───────────────────
labels = pd.read_csv(ROOT / "train_labels.csv", keep_default_na=False, dtype=str)
labels["ocr_text"]     = labels["ocr_text"].astype(str).str.strip()
labels["product_name"] = labels["product_name"].astype(str).str.strip()

test_ids = pd.read_csv(ROOT / "test.csv", keep_default_na=False, dtype=str)
sample   = pd.read_csv(ROOT / "sample_submission.csv", keep_default_na=False, dtype=str)

print(f"train rows : {len(labels):,}")
print(f"test  rows : {len(test_ids):,}")
print(f"empty ocr_text     : {(labels.ocr_text=='').mean():.1%}")
print(f"empty product_name : {(labels.product_name=='').mean():.1%}")
print("\nTop product families:")
print(labels[labels.product_name != ''].product_name.value_counts().head(8).to_string())


# ── Exact competition metric ────────────────────────────────────────────────────
def _clean(v):
    return "" if pd.isna(v) else str(v).strip()

def token_f1(gt, pred):
    """Case-insensitive token-set F1. both-empty -> 1.0 ; one-empty -> 0.0."""
    gt, pred = _clean(gt), _clean(pred)
    if not gt and not pred:
        return 1.0
    g, p = set(gt.lower().split()), set(pred.lower().split())
    if not g or not p:
        return 0.0
    common = g & p
    if not common:
        return 0.0
    precision = len(common) / len(p)
    recall    = len(common) / len(g)
    return 2 * precision * recall / (precision + recall)

def cer(gt, pred):
    """Char error rate = Levenshtein(pred, gt) / len(gt), clamped to 1.0."""
    gt, pred = _clean(gt), _clean(pred)
    if len(gt) == 0:
        return 0.0 if len(pred) == 0 else 1.0
    m, n = len(gt), len(pred)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            tmp = dp[j]
            dp[j] = prev if gt[i-1] == pred[j-1] else 1 + min(prev, dp[j], dp[j-1])
            prev = tmp
    return min(dp[n] / len(gt), 1.0)

def composite(gt_ocr, pred_ocr, gt_prod, pred_prod):
    f1  = float(np.mean([token_f1(g, p) for g, p in zip(gt_prod, pred_prod)]))
    avg = float(np.mean([cer(g, p)      for g, p in zip(gt_ocr,  pred_ocr)]))
    ot  = 1.0 - avg
    return {"composite": round(W_PRODUCT_F1*f1 + W_OCR*ot, 4),
            "product_f1": round(f1, 4), "ocr_term": round(ot, 4), "cer": round(avg, 4)}

print("\nmetric self-check:",
      "token_f1('a b','a b')=", token_f1('a b','a b'),
      "| cer('abc','abx')=", round(cer('abc','abx'), 3))

## §2  OCR engine — fine-tuned VietOCR (`vgg_transformer` + CRAFT)

The OCR stage produces `ocr_text` (scored by CER). Our recogniser is **VietOCR `vgg_transformer`** (37.9M params) fine-tuned on this competition's images, with the **CRAFT** detector (via EasyOCR) for line detection. We chose VietOCR because CER is **diacritic-sensitive** and VietOCR is the only engine that reliably preserves Vietnamese tone marks (diac density 0.215 vs PaddleOCR 0.024; GT reference 0.285).

**Fine-tuning procedure** (Cell below, `RUN_OCR_FROM_SCRATCH = True`):
1. Detect text lines on each train image (CRAFT), crop them.
2. Recognise each crop with **base** VietOCR, fuzzy-align the prediction to the matching span of the GT `ocr_text` → `(crop, correct-diacritic label)` pairs.
3. Fine-tune VietOCR on those pairs (15k iters).
4. Run the fine-tuned model on test + train images → `ocr_text` parquet.

If `RUN_OCR_FROM_SCRATCH = False`, the next cell is skipped and we load the precomputed parquet instead (recommended for a quick review — the fine-tune takes 2–4 h on a T4).

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
#  §2a  FINE-TUNE VietOCR FROM RAW IMAGES  (only runs if RUN_OCR_FROM_SCRATCH=True)
#  Needs: GPU T4 + Internet ON.  Runtime ~2-4 h.  Mirrors kaggle_vietocr_finetune.ipynb
# ════════════════════════════════════════════════════════════════════════════════
if RUN_OCR_FROM_SCRATCH:
    import subprocess, sys, time, random, cv2
    from PIL import Image
    from tqdm.auto import tqdm

    # deps (pin Pillow last to avoid ImageFont ImportError) — restart kernel if it warns
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "easyocr", "vietocr", "rapidfuzz"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "--force-reinstall", "Pillow==10.4.0"], check=False)

    import easyocr
    from rapidfuzz import fuzz
    from vietocr.tool.predictor import Predictor
    from vietocr.tool.config import Cfg

    # locate image dirs
    def _imgdir(*cands):
        for rel in cands:
            d = ROOT / rel
            if d.is_dir() and any(d.glob("*.jpg")):
                return d
        raise FileNotFoundError(cands)
    TRAIN_DIR = _imgdir("train_images/train_images", "train_images")
    TEST_DIR  = _imgdir("test_images/images", "images", "test_images")

    def fold_ocr(s):
        s = unicodedata.normalize("NFD", str(s).lower())
        s = "".join(c for c in s if unicodedata.category(c) != "Mn").replace(chr(0x111), "d")
        return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", s)).strip()

    def clean_ocr(t, maxlen=MAX_OCR_LEN):
        t = unicodedata.normalize("NFC", str(t)).replace(chr(10), " ").replace(chr(9), " ")
        t = re.sub(r"\s+", " ", t).strip()
        toks = t.split(); ded = [toks[0]] if toks else []
        for w in toks[1:]:
            if w.lower() != ded[-1].lower(): ded.append(w)
        t = " ".join(ded)
        return t[:maxlen].rstrip() if len(t) > maxlen else t

    detector = easyocr.Reader(["vi", "en"], gpu=True, verbose=False)   # CRAFT detector
    base_cfg = Cfg.load_config_from_name("vgg_transformer")
    base_cfg["device"] = "cuda:0"; base_cfg["predictor"]["beamsearch"] = False
    base_rec = Predictor(base_cfg); VOCAB = set(base_cfg["vocab"])
    LINES = WORK / "lines"; LINES.mkdir(exist_ok=True)

    # 1) build aligned (crop -> GT span) pairs
    def best_gt_span(pred, gt_tokens, max_span=10):
        pf = fold_ocr(pred)
        if len(pf) < 2: return None, 0
        best, bs, n = None, 0, len(gt_tokens)
        for i in range(n):
            for L in range(1, max_span + 1):
                if i + L > n: break
                span = " ".join(gt_tokens[i:i+L])
                sc = fuzz.ratio(pf, fold_ocr(span))
                if sc > bs: bs, best = sc, span
            if bs == 100: break
        return best, bs

    rows = labels[labels.ocr_text != ""].copy()
    pairs, idx = [], 0
    for _, r in tqdm(rows.iterrows(), total=len(rows), desc="align"):
        img = cv2.imread(str(TRAIN_DIR / (r.image_id + ".jpg")))
        if img is None: continue
        gt_tokens = r.ocr_text.split()
        try:
            horiz, _ = detector.detect(img, min_size=15, text_threshold=0.6)
            boxes = horiz[0] if horiz else []
        except Exception:
            boxes = []
        for b in boxes:
            x0, x1, y0, y1 = [int(v) for v in b]
            x0, y0 = max(0, x0), max(0, y0)
            x1, y1 = min(img.shape[1], x1), min(img.shape[0], y1)
            if x1 - x0 < 6 or y1 - y0 < 6: continue
            crop = img[y0:y1, x0:x1]
            try:
                pred = base_rec.predict(Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)))
            except Exception:
                continue
            span, sc = best_gt_span(pred, gt_tokens)
            if span is None or sc < 75: continue
            label = "".join(ch for ch in span if ch in VOCAB).strip()
            if len(label) < 1: continue
            fn = "l%06d.png" % idx; cv2.imwrite(str(LINES / fn), crop); idx += 1
            pairs.append((fn, label))
    print("built", len(pairs), "aligned pairs")
    random.seed(0); random.shuffle(pairs)
    with open(LINES / "train_ann.txt", "w", encoding="utf-8") as f:
        for fn, lb in pairs: f.write(fn + "\t" + lb + "\n")

    # 2) fine-tune (numpy-2.0 shims for vietocr/imgaug)
    if not hasattr(np, "sctypes"):
        np.sctypes = {"int":[np.int8,np.int16,np.int32,np.int64],
                      "uint":[np.uint8,np.uint16,np.uint32,np.uint64],
                      "float":[np.float16,np.float32,np.float64],
                      "complex":[np.complex64,np.complex128],
                      "others":[bool,object,bytes,str,np.void]}
    for _n,_t in [("bool",bool),("float",float),("int",int),("object",object),("str",str),("complex",complex)]:
        if not hasattr(np,_n): setattr(np,_n,_t)
    from vietocr.model.trainer import Trainer
    cfg = Cfg.load_config_from_name("vgg_transformer")
    cfg["device"] = "cuda:0"; cfg["dataloader"] = {"num_workers": 2}
    cfg["dataset"] = {"name": "ura_%d" % int(time.time()), "data_root": str(LINES),
                      "train_annotation": "train_ann.txt", "valid_annotation": None,
                      "image_height": 32, "image_min_width": 32, "image_max_width": 512}
    cfg["trainer"] = {"batch_size": 32, "print_every": 200, "valid_every": 10**9,
                      "iters": 15000, "export": "./vietocr_ft.pth", "checkpoint": "./ckpt.pth",
                      "log": "./train.log", "metrics": 5000}
    Trainer(cfg, pretrained=True).train()
    Trainer(cfg, pretrained=True).save_weights("./vietocr_ft.pth")

    # 3) inference with fine-tuned model -> parquet (run_ocr.py schema)
    ft_cfg = Cfg.load_config_from_name("vgg_transformer")
    ft_cfg["device"] = "cuda:0"; ft_cfg["predictor"]["beamsearch"] = False
    ft_cfg["weights"] = "./vietocr_ft.pth"
    ft_rec = Predictor(ft_cfg)

    def ocr_image(path):
        img = cv2.imread(str(path))
        if img is None: return {"ocr_text": "", "n_boxes": 0, "n_chars": 0, "mean_conf": 0.0}
        try:
            horiz, _ = detector.detect(img, min_size=15, text_threshold=0.6)
            boxes = horiz[0] if horiz else []
        except Exception:
            boxes = []
        items = []
        for b in boxes:
            x0, x1, y0, y1 = [int(v) for v in b]
            x0, y0 = max(0, x0), max(0, y0)
            x1, y1 = min(img.shape[1], x1), min(img.shape[0], y1)
            if x1 - x0 < 6 or y1 - y0 < 6: continue
            items.append((y0, x0, Image.fromarray(cv2.cvtColor(img[y0:y1, x0:x1], cv2.COLOR_BGR2RGB))))
        if not items: return {"ocr_text": "", "n_boxes": 0, "n_chars": 0, "mean_conf": 0.0}
        try: texts = ft_rec.predict_batch([it[2] for it in items])
        except Exception: texts = [ft_rec.predict(it[2]) for it in items]
        ys = [it[0] for it in items]; band = max(8.0, (max(ys)-min(ys))/40.0)
        order = sorted(range(len(items)), key=lambda i: (round(items[i][0]/band), items[i][1]))
        text = clean_ocr(" ".join(texts[i] for i in order if str(texts[i]).strip()))
        return {"ocr_text": text, "n_boxes": len(items), "n_chars": len(text), "mean_conf": 0.0}

    def run_ocr(ids, img_dir, out, save_every=200):
        recs = []
        for k, iid in enumerate(tqdm(ids, desc=out.name)):
            r = ocr_image(img_dir / (iid + ".jpg")); r["image_id"] = iid; recs.append(r)
            if (k + 1) % save_every == 0: pd.DataFrame(recs).to_parquet(out, index=False)
        pd.DataFrame(recs).to_parquet(out, index=False)
        return out

    OCR_TEST_PARQUET = run_ocr(test_ids.image_id.tolist(), TEST_DIR,
                               WORK / f"ocr_{OCR_ENGINE_TAG}_test.parquet")
    OCR_ALL_PARQUET  = run_ocr(labels.image_id.tolist(), TRAIN_DIR,
                               WORK / f"ocr_{OCR_ENGINE_TAG}_all.parquet")
    print("OCR parquets written:", OCR_ALL_PARQUET, OCR_TEST_PARQUET)
else:
    print("RUN_OCR_FROM_SCRATCH = False  ->  skipping fine-tune; will load cached parquet.")

In [ ]:
# ── Load the VietOCR-FT outputs (train+test) into dataframes ────────────────────
assert OCR_ALL_PARQUET and Path(OCR_ALL_PARQUET).exists(),  "missing OCR (all) parquet"
assert OCR_TEST_PARQUET and Path(OCR_TEST_PARQUET).exists(), "missing OCR (test) parquet"

ocr_all  = pd.read_parquet(OCR_ALL_PARQUET)
ocr_test = pd.read_parquet(OCR_TEST_PARQUET)
for d in (ocr_all, ocr_test):
    d["ocr_text"] = d["ocr_text"].fillna("")
    if "n_boxes"  not in d: d["n_boxes"]  = 0
    if "mean_conf" not in d: d["mean_conf"] = 0.0
    if "n_chars"  not in d: d["n_chars"]  = d["ocr_text"].str.len()

# diacritic density sanity check (should be ~0.18-0.22, NOT ~0.02 like PaddleOCR)
def _diac_frac(s):
    nfd = unicodedata.normalize("NFD", str(s))
    letters = sum(c.isalpha() for c in nfd)
    marks   = sum(unicodedata.category(c) == "Mn" for c in nfd)
    return marks / letters if letters else 0.0

print(f"OCR (all)  rows : {len(ocr_all):,}")
print(f"OCR (test) rows : {len(ocr_test):,}")
print(f"test OCR fill   : {(ocr_test.ocr_text.str.strip()!='').mean():.1%}")
print(f"engine diac density (train) : {ocr_all.ocr_text.map(_diac_frac).mean():.3f}  "
      f"(GT ref ~0.285 — confirms diacritics preserved)")

## §3  Product head — `CalibratedRuleHead`

Produces `product_name` (scored by token-F1). Three embedded components:

1. **`ProductExtractor`** — TF-IDF char n-gram + LogisticRegression, with a binary "has-product" gate and modal-form canonicalisation. This is the long-tail fallback.
2. **`CalibratedRuleHead`** — 9 ordered family signatures (most-specific first). For each family it computes, **on the training fold only**, the emit string that maximises expected token-F1 over that cluster's GT labels (the *emit-gain* rule). A rule deploys only if its train precision clears `min_pprod=0.55`. Unmatched images fall through to the classifier.
3. **`EmptyGate`** — logistic gate on OCR features (box count, char count) that zeroes `ocr_text` for textless images (CER 1.0 → 0.0).

The `fit()` methods recompute everything from their input data, so they are **fold-safe**: in CV (§4) they see only training-fold labels, never val.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

try:
    from unidecode import unidecode
except Exception:
    def unidecode(s): return s


# ── shared text folding ─────────────────────────────────────────────────────────
def fold_uni(s):
    """Diacritic-insensitive, lowercased, alnum-only — for classifier features."""
    s = unidecode(str(s)).lower()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", s)).strip()

def fold_nfd(s):
    """NFD-based fold used by the rule patterns (handles đ -> d)."""
    s = unicodedata.normalize("NFD", str(s).lower()).replace("đ", "d")
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", s)).strip()

def tokkey(s):
    return " ".join(sorted(str(s).lower().split()))


# ── (1) long-tail classifier ────────────────────────────────────────────────────
class ProductExtractor:
    def __init__(self, min_class_count=3, gate_threshold=0.5, max_features=5000):
        self.min_class_count = min_class_count
        self.gate_threshold  = gate_threshold
        self.max_features    = max_features
        self.canonical, self._gate, self._clf, self.classes_ = {}, None, None, []

    def _canonicalize(self, df):
        nonempty = df[df.product_name != ""]
        canon = {}
        for key, grp in nonempty.groupby(nonempty.product_name.map(tokkey)):
            canon[key] = grp.product_name.value_counts().idxmax()
        self.canonical = canon
        return df.product_name.map(lambda p: canon.get(tokkey(p), "") if p else "")

    def fit(self, df):
        df = df[["ocr_text", "product_name"]].copy()
        df["ocr_text"]     = df["ocr_text"].astype(str).str.strip()
        df["product_name"] = df["product_name"].astype(str).str.strip()
        df["folded"] = df["ocr_text"].map(fold_uni)
        df["canon"]  = self._canonicalize(df)
        self._gate = Pipeline([
            ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5),
                                      max_features=self.max_features, min_df=2)),
            ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", C=4.0))])
        self._gate.fit(df["folded"], (df["canon"] != "").astype(int))
        pos = df[(df.folded != "") & (df.canon != "")]
        keep = pos["canon"].value_counts()
        keep = keep[keep >= self.min_class_count].index
        pos = pos[pos["canon"].isin(keep)]
        self.classes_ = sorted(pos["canon"].unique())
        self._clf = Pipeline([
            ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5),
                                      max_features=self.max_features, min_df=2)),
            ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", C=4.0))])
        if len(pos):
            self._clf.fit(pos["folded"], pos["canon"])
        return self

    def predict(self, ocr_text):
        t = "" if ocr_text is None else str(ocr_text).strip()
        if not t: return ""
        f = fold_uni(t)
        if not f or self._gate is None: return ""
        classes = list(self._gate.classes_)
        if 1 not in classes: return ""
        p1 = self._gate.predict_proba([f])[0][classes.index(1)]
        if p1 < self.gate_threshold: return ""
        if self._clf is None or not self.classes_: return ""
        return str(self._clf.predict([f])[0])

    def predict_batch(self, texts):
        return [self.predict(t) for t in texts]


# ── (2) calibrated dominant-family rule head ────────────────────────────────────
SIG_PATTERNS = [
    ("halong_canfoco_pate_cotden",
     r"(canfoco|canfuco|cafoco).*(pate|cot den).*(cot den|hai phong)|(pate|cot den).*(canfoco|canfuco)"),
    ("nan_optipro",   r"\bnan\b.*opti ?pro|opti ?pro.*\bnan\b"),
    ("nan_infinipro", r"\bnan\b.*infini ?pro|infini ?pro.*\bnan\b"),
    ("pate_cotden",   r"\bpate\b.*\b(cot den|hai phong)\b|\b(cot den|hai phong)\b.*\bpate\b|\bcot den\b"),
    ("halong_canfoco", r"\bcanfoco\b|\bcanfuco\b|\bcafoco\b|ha long canfoco|halong canfoco"),
    ("do_hop_ha_long", r"do hop ha long|do hop.*ha long|cong ty.*do hop.*ha long"),
    ("nan",           r"\bnan\b"),
    ("highlands",     r"highlands? coffee|highlands"),
    ("nestle",        r"\bnestle\b"),
]

class CalibratedRuleHead:
    def __init__(self, use_classifier_fallback=True, min_pprod=MIN_PPROD,
                 min_class_count=MIN_CLASS_COUNT, gate_threshold=GATE_THRESHOLD):
        self.use_clf   = use_classifier_fallback
        self.min_pprod = min_pprod
        self._clf  = ProductExtractor(min_class_count=min_class_count,
                                      gate_threshold=gate_threshold)
        self.rules = []   # (name, compiled_pat, emit_form, p_prod)

    def fit(self, df):
        df = df.copy()
        df["folded_ocr"] = df["ocr_text"].map(fold_nfd)
        df["prod"] = df["product_name"].astype(str).str.strip()
        remaining = df
        self.rules = []
        for name, pat in SIG_PATTERNS:
            m = remaining["folded_ocr"].str.contains(pat, regex=True, na=False)
            sub = remaining[m]
            if len(sub) >= 8:
                p_prod = (sub["prod"] != "").mean()
                forms = Counter(sub[sub["prod"] != ""]["prod"])
                cands = [f for f, c in forms.items() if c >= 3] or list(forms)
                gts = list(sub["prod"])
                best_form, best_val = "", -1.0
                for cand in cands:                       # emit-gain: argmax E[token_f1]
                    v = float(np.mean([token_f1(g, cand) for g in gts]))
                    if v > best_val:
                        best_val, best_form = v, cand
                empty_base = (sub["prod"] == "").mean()
                if best_form and (best_val - empty_base) > 0 and p_prod >= self.min_pprod:
                    self.rules.append((name, re.compile(pat), best_form, p_prod))
            remaining = remaining[~m]                    # sequential consumption
        if self.use_clf:
            self._clf.fit(df[["image_id", "ocr_text", "product_name"]]
                          if "image_id" in df else df)
        return self

    def _rule(self, folded):
        for _name, pat, form, _p in self.rules:
            if pat.search(folded):
                return form
        return ""

    def predict(self, ocr_text):
        t = "" if ocr_text is None else str(ocr_text).strip()
        if not t: return ""
        hit = self._rule(fold_nfd(t))
        if hit: return hit
        return self._clf.predict(t) if self.use_clf else ""

    def predict_batch(self, texts):
        return [self.predict(t) for t in texts]


# ── (3) empty-gate ──────────────────────────────────────────────────────────────
EG_FEATURES = ["log_boxes", "mean_conf", "log_chars"]

def _eg_featurize(df):
    df = df.copy()
    n_chars = df.get("n_chars")
    if n_chars is None:
        n_chars = df["ocr_text"].fillna("").str.len()
    df["log_boxes"] = np.log1p(df["n_boxes"].fillna(0))
    df["log_chars"] = np.log1p(pd.Series(n_chars, index=df.index).fillna(0))
    df["mean_conf"] = df["mean_conf"].fillna(0.0)
    return df

class EmptyGate:
    def __init__(self, threshold=EMPTY_GATE_THR):
        self.threshold = threshold; self.clf = None
    def fit(self, ocr_features, gt_empty):
        X = _eg_featurize(ocr_features)[EG_FEATURES]
        self.clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X, gt_empty)
        return self
    def is_empty(self, ocr_features):
        if self.clf is None:
            return np.zeros(len(ocr_features), dtype=bool)
        X = _eg_featurize(ocr_features)[EG_FEATURES]
        return self.clf.predict_proba(X)[:, 1] >= self.threshold

print("Product head, classifier, and empty-gate defined.")

## §4  Cross-validation — 5-fold GroupKFold (the trustworthy estimate)

We group by the **MD5 of the normalised GT `ocr_text`** so near-duplicate thumbnails (same text, different crop) never straddle train/val — this prevents leakage that would inflate the score. Empty-text rows each get a unique group so they spread proportionally.

Inside every fold the product head, classifier, and empty-gate are **refit on the training fold only**, then evaluated on the held-out fold's **real VietOCR-FT output** (matching the deployed pipeline exactly). The OCR/CER half uses the cached OCR (fixed per image, no training).

**Expected result: pooled composite ≈ 0.6142** — this is the honest CV number behind the 0.6685 public LB (the +0.054 gap is train→test distribution shift, not overfitting; we never tuned on the public board).

In [ ]:
from sklearn.model_selection import GroupKFold

def _group_key(text, image_id):
    t = " ".join(str(text).lower().split())
    return "empty_" + str(image_id) if not t else hashlib.md5(t.encode("utf-8")).hexdigest()

# merge GT labels with the cached OCR (suffix _gt = label, _ocr = engine output)
cv_df = labels.merge(ocr_all, on="image_id", suffixes=("_gt", "_ocr")).reset_index(drop=True)
cv_df["ocr_text_ocr"] = cv_df["ocr_text_ocr"].fillna("")
groups = [_group_key(t, i) for i, t in zip(cv_df.image_id, cv_df.ocr_text_gt)]

gkf = GroupKFold(n_splits=N_FOLDS)
pooled_gt, pooled_pred, pooled_cer, fold_scores = [], [], [], []

for fold_i, (tr_idx, va_idx) in enumerate(gkf.split(cv_df, groups=groups), 1):
    trd, vad = cv_df.iloc[tr_idx], cv_df.iloc[va_idx]

    # product head: fit on TRAIN fold's GT text only (fold-safe)
    head = CalibratedRuleHead(use_classifier_fallback=True, min_pprod=MIN_PPROD,
                              min_class_count=MIN_CLASS_COUNT, gate_threshold=GATE_THRESHOLD)
    head.fit(trd.rename(columns={"ocr_text_gt": "ocr_text"})
                [["image_id", "ocr_text", "product_name"]])

    # empty-gate: fit on TRAIN fold's OCR features only (fold-safe)
    tg = trd.copy(); tg["gt_empty"] = (tg.ocr_text_gt.str.strip() == "").astype(int)
    eg = EmptyGate(threshold=EMPTY_GATE_THR).fit(
        tg.rename(columns={"ocr_text_ocr": "ocr_text"}), tg.gt_empty)
    mask = eg.is_empty(vad.rename(columns={"ocr_text_ocr": "ocr_text"}))
    ocr_in = vad.ocr_text_ocr.where(~np.asarray(mask), "")

    # predict product on the held-out fold's REAL OCR text
    preds = head.predict_batch(ocr_in)
    f1s = [token_f1(g, p) for g, p in zip(vad.product_name, preds)]
    cers = [cer(g, p)      for g, p in zip(vad.ocr_text_gt, ocr_in)]
    fold_scores.append(0.6 * np.mean(f1s) + 0.4 * (1 - np.mean(cers)))
    pooled_pred += list(preds); pooled_gt += list(vad.product_name); pooled_cer += cers
    print(f"  fold {fold_i}: composite={fold_scores[-1]:.4f}  "
          f"(F1={np.mean(f1s):.4f}, ocr_term={1-np.mean(cers):.4f}, rules={len(head.rules)})")

cv_f1  = float(np.mean([token_f1(g, p) for g, p in zip(pooled_gt, pooled_pred)]))
cv_ot  = 1 - float(np.mean(pooled_cer))
cv_comp = 0.6 * cv_f1 + 0.4 * cv_ot
print(f"\n{'='*52}")
print(f"  POOLED 5-fold CV composite : {cv_comp:.4f}")
print(f"  product F1 : {cv_f1:.4f}   ocr_term : {cv_ot:.4f}")
print(f"  fold mean  : {np.mean(fold_scores):.4f}   fold std : {np.std(fold_scores):.4f}")
print(f"  (public LB of this exact config = 0.6685; gap = distribution shift)")
print(f"{'='*52}")

## §5  Train the final head on ALL labels

For the submission (not CV) there is no held-out fold — the head and empty-gate are fit on **all** training labels to use every available example. This is the exact model that produced `submission_v10_calibrated.csv`.

In [ ]:
# ── Final product head: fit on ALL training labels (clean GT text) ──────────────
final_head = CalibratedRuleHead(use_classifier_fallback=True, min_pprod=MIN_PPROD,
                                min_class_count=MIN_CLASS_COUNT, gate_threshold=GATE_THRESHOLD)
final_head.fit(labels[["image_id", "ocr_text", "product_name"]])

print(f"Deployed {len(final_head.rules)} calibrated rules (most-specific first):")
for name, _pat, form, p in final_head.rules:
    print(f"  {name:<28} p_prod={p:.3f}  ->  '{form}'")

# ── Final empty-gate: fit on ALL train OCR features ─────────────────────────────
feats = labels.merge(ocr_all, on="image_id", suffixes=("_lab", "_ocr"))
feats["gt_empty"] = (feats["ocr_text_lab"].fillna("").str.strip() == "").astype(int)
final_gate = EmptyGate(threshold=EMPTY_GATE_THR).fit(
    feats.rename(columns={"ocr_text_ocr": "ocr_text"}), feats["gt_empty"])
print("\nFinal head + empty-gate trained on all", len(labels), "labels.")

## §6  Inference on the test set

Take the cached VietOCR-FT test OCR, apply the empty-gate (zero textless images), then predict `product_name` with the final head. Order rows to match `sample_submission.csv`.

In [ ]:
# ── Assemble test predictions ───────────────────────────────────────────────────
sub = test_ids.merge(ocr_test, on="image_id", how="left")
sub["ocr_text"] = sub["ocr_text"].fillna("")
if "mean_conf" not in sub: sub["mean_conf"] = 0.0
if "n_boxes"  not in sub: sub["n_boxes"]  = 0

n_missing = len(set(test_ids.image_id) - set(ocr_test.image_id))
if n_missing:
    print(f"  WARNING: {n_missing} test images missing from OCR cache (forced empty).")

# empty-gate: zero OCR text on predicted-textless images
mask = final_gate.is_empty(sub)
sub.loc[mask, "ocr_text"] = ""
print(f"empty-gate zeroed {int(mask.sum())} test OCR rows")

# product prediction on the (possibly gated) OCR text
sub["product_name"] = final_head.predict_batch(sub["ocr_text"])
sub = sub[["image_id", "ocr_text", "product_name"]]

print(f"\nrows={len(sub)} | OCR fill={(sub.ocr_text.str.strip()!='').mean():.1%} | "
      f"product fill={(sub.product_name.str.strip()!='').mean():.1%}")
print("\nTop emitted products:")
print(sub[sub.product_name.str.strip()!=''].product_name.value_counts().head(10).to_string())

## §7  Build & validate `submission.csv`

Run the competition format checks (AC-1…AC-7), replace blank cells with a single space (Kaggle rejects empty cells), and write the UTF-8 `QUOTE_ALL` CSV that Kaggle accepts. This file is the deliverable — upload it to the competition's *Submit Predictions* page.

In [ ]:
# ── Validate against sample_submission (AC-1..AC-7) ─────────────────────────────
def validate(sub_df, sample_df):
    expected, got = set(sample_df.image_id), set(sub_df.image_id)
    checks = {
        "AC-1 row count":    len(sub_df) == len(sample_df),
        "AC-2 no extra ids": len(got - expected) == 0,
        "AC-3 no missing ids": len(expected - got) == 0,
        "AC-4 no dup ids":   not sub_df.image_id.duplicated().any(),
        "AC-5 no nulls":     not sub_df[["image_id","ocr_text","product_name"]].isnull().any().any(),
        "AC-6 no newline/tab": not sub_df.ocr_text.str.contains(r"[\n\t]", regex=True, na=False).any(),
        "AC-7 columns":      list(sub_df.columns[:3]) == ["image_id","ocr_text","product_name"],
    }
    ok = True
    for nm, passed in checks.items():
        print(f"  [{'PASS' if passed else 'FAIL'}] {nm}"); ok = ok and passed
    return ok

assert validate(sub, sample), "submission validation FAILED"

# reorder to match sample submission
sub = sub.set_index("image_id").reindex(sample["image_id"]).reset_index()

# blank -> single space (Kaggle rejects empty cells), write UTF-8 QUOTE_ALL
out = sub[["image_id", "ocr_text", "product_name"]].copy()
for col in ("ocr_text", "product_name"):
    out[col] = out[col].fillna("").astype(str).str.strip()
    out.loc[out[col] == "", col] = " "

OUT_PATH = WORK / "submission.csv"
out.to_csv(OUT_PATH, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
print(f"\nWrote {OUT_PATH}  ({len(out)} rows)")
print(f"This is the v10_calibrated submission (public LB 0.6685). Upload to Kaggle.")
out.head()